# Camada Bronze

Este notebook executa as funções de ingestão dos arquivos locais e grava os resultados em Parquet, em `data/bronze`.

Execute as células em ordem, usando o kernel do ambiente virtual do projeto. O diretório de trabalho deve ser a raiz do projeto ou uma de suas subpastas.

Não há consultas a APIs nem carga no banco. Os módulos preservam os dados brutos e acrescentam informações de origem. Cada ingestão usa o modo overwrite: se o destino já existir, os arquivos de saída dessa fonte são substituídos pelos dados da nova execução. Os arquivos de entrada são mantidos.

As fontes por UF usam pastas como `SP` e arquivos como `municipalities_SP.parquet`. Quando o Spark produz vários arquivos, eles são numerados. A UF também é mantida como coluna nos arquivos. Não são gerados marcadores `_SUCCESS`; os arquivos ocultos `.crc` verificam a integridade dos Parquets.

Para ler uma fonte inteira com pastas de UF simples, use `spark.read.option("recursiveFileLookup", "true").parquet(caminho)`. Os campos aninhados são preservados; a célula de municípios inclui uma visualização explícita de São Paulo.


A substituição usa uma carga temporária validada, com recuperação da carga anterior em caso de falha. As pastas auxiliares são removidas ao terminar; não são mantidas versões de cada execução.

A gravação usa uma trava do sistema operacional e um diário de recuperação. Após encerramento brusco do processo, a próxima gravação recupera a operação pendente automaticamente. Para recuperar antes de consultar, use `pipeline.shared.storage.recover_output(caminho)`. O pequeno arquivo `.write-lock` é reutilizado e não precisa ser apagado.


O acompanhamento usa somente o estado atual. Os valores permanecem nos arquivos das camadas; não são criados snapshots. Para atualizar os vínculos no PostgreSQL, execute a Gold e depois o Load.


## 1. Imports e localização do projeto

O caminho é identificado pelo diretório de trabalho do notebook, sem usar `__file__`, que não está disponível em notebooks.


In [ ]:
import sys
from pathlib import Path
import os
from importlib import reload

from pyspark.sql import SparkSession
from pyspark.sql import functions as F


project_directory = Path.cwd().resolve()

# Procura a raiz do projeto a partir do diretório atual.
while not (project_directory / "pipeline" / "bronze").is_dir() or not (project_directory / "data").is_dir():

    if project_directory == project_directory.parent:
        raise FileNotFoundError("Abra o notebook com o diretório de trabalho dentro do projeto Viajando_com_o_dado.")

    project_directory = project_directory.parent

# Prioriza os módulos da Bronze, pois a coleta tem arquivos com os mesmos nomes.
bronze_modules_directory = project_directory / "pipeline" / "bronze"
sys.path.insert(0, str(project_directory))
sys.path.insert(0, str(bronze_modules_directory))

from pipeline.shared import execution
import geocoding
import ibge_municipalities
import ibge_population
import inmet
import open_meteo
import bronze_storage


DATA_DIRECTORY = project_directory / "data"
BRONZE_DIRECTORY = DATA_DIRECTORY / "bronze"
YEAR = "2025"

print(f"Projeto: {project_directory}")
print(f"Destino da Bronze: {BRONZE_DIRECTORY}")


In [ ]:
session = execution.run_context(project_directory, parameters={"year": str(globals().get("YEAR", "2025"))})
state = session.__enter__()
print("Acompanhando a execução atual.")


## 2. Módulos utilizados

Os imports estão reunidos na primeira célula de código, após a configuração dos caminhos quando necessária.


## 3. Iniciar o Spark

A sessão será compartilhada pelas cinco ingestões. Esta célula exige o PySpark e uma instalação de Java compatível no ambiente.

Para executar localmente no Windows, o projeto inclui `lib/hadoop-bare-naked-local-fs-0.1.0.jar`, da GlobalMentor, obtido do Maven Central com checksum verificado. A biblioteca usa as APIs de arquivos do Java, sem depender de `winutils.exe` ou `hadoop.dll`.

Após alterar essa configuração, reinicie o kernel antes de executar o notebook. O Spark pode exibir avisos sobre a ausência de bibliotecas nativas; a leitura e a gravação locais usam a implementação Java configurada abaixo.

Referência: [GlobalMentor Hadoop Bare Naked Local FileSystem](https://github.com/globalmentor/hadoop-bare-naked-local-fs).


In [ ]:
# Usa o Python do kernel também nos processos de trabalho do Spark.
os.environ["PYSPARK_PYTHON"] = sys.executable

# A biblioteca Java permite acessar arquivos locais no Windows sem winutils ou hadoop.dll.
local_filesystem_jar = bronze_modules_directory / "lib" / "hadoop-bare-naked-local-fs-0.1.0.jar"

if not local_filesystem_jar.is_file():
    raise FileNotFoundError(f"Biblioteca de acesso local não encontrada: {local_filesystem_jar}")

spark = (
    SparkSession.builder
    .appName("Viajando com o dado - Bronze")
    .master("local[2]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.hadoop.mapreduce.fileoutputcommitter.marksuccessfuljobs", "false")
    .config("spark.pyspark.python", sys.executable)
    .config("spark.driver.extraClassPath", str(local_filesystem_jar))
    .config("spark.executor.extraClassPath", str(local_filesystem_jar))
    .config("spark.hadoop.fs.file.impl", "com.globalmentor.apache.hadoop.fs.BareLocalFileSystem")
    .getOrCreate()
)

print(f"Versão do Spark: {spark.version}")


## 4. Municípios do IBGE

Lê os JSONs por UF e preserva as hierarquias regionais. A coleta atual contém apenas as capitais.

Entrada: `data/ibge/municipalities`  
Saída: `data/bronze/ibge/municipalities`


In [ ]:
# Atualiza também a rotina compartilhada de gravação.
reload(bronze_storage)
reload(ibge_municipalities)

municipalities = ibge_municipalities.ingest_municipalities(
    spark,
    DATA_DIRECTORY / "ibge" / "municipalities",
    BRONZE_DIRECTORY / "ibge" / "municipalities",
)

municipalities.printSchema()

# Exibe os campos aninhados explicitamente, pois alguns visualizadores os mostram vazios.
municipalities.filter(F.col("state_code") == "SP").select(
    "id",
    "nome",
    F.col("microrregiao.id").alias("microregion_id"),
    F.col("microrregiao.nome").alias("microregion_name"),
    "state_code",
).show(truncate=False)


## 5. População do IBGE

Preserva as linhas da aba `Municípios`, incluindo título, cabeçalho e notas. Os valores de cada linha ficam em `row_values_json`. A seleção das capitais será feita na Silver.

Entrada: `data/ibge/population`  
Saída: `data/bronze/ibge/population`


In [ ]:
# Atualiza também a rotina compartilhada de gravação.
reload(bronze_storage)
reload(ibge_population)

population = ibge_population.ingest_population(
    spark,
    DATA_DIRECTORY / "ibge" / "population",
    BRONZE_DIRECTORY / "ibge" / "population",
)

population.printSchema()


## 6. Coordenadas das capitais

Preserva as coordenadas consultadas e identifica a UF pelo nome do arquivo.

Entrada: `data/open_meteo/geocoding`  
Saída: `data/bronze/open_meteo/geocoding`


In [ ]:
# Atualiza também a rotina compartilhada de gravação.
reload(bronze_storage)
reload(geocoding)

coordinates = geocoding.ingest_geocoding(
    spark,
    DATA_DIRECTORY / "open_meteo" / "geocoding",
    BRONZE_DIRECTORY / "open_meteo" / "geocoding",
)

coordinates.printSchema()


## 7. Clima das capitais — Open-Meteo

Mantém os metadados, as unidades e as listas diárias da resposta. A organização em uma linha por dia fica para a Silver.

Entrada: `data/open_meteo/weather/2025`  
Saída: `data/bronze/open_meteo/weather/2025`


In [ ]:
# Atualiza também a rotina compartilhada de gravação.
reload(bronze_storage)
reload(open_meteo)

weather = open_meteo.ingest_open_meteo(
    spark,
    DATA_DIRECTORY / "open_meteo" / "weather" / YEAR,
    BRONZE_DIRECTORY / "open_meteo" / "weather" / YEAR,
)

weather.printSchema()


## 8. Estações do INMET

Lê todos os CSVs da pasta do ano, preservando metadados, cabeçalhos e medições como texto. Esta etapa inclui as 594 estações disponíveis; ainda não seleciona as estações das capitais.

Entrada: `data/inmet/2025`  
Saída: `data/bronze/inmet/2025`


In [ ]:
# Atualiza também a rotina compartilhada de gravação.
reload(bronze_storage)
reload(inmet)

station_observations = inmet.ingest_inmet(
    spark,
    DATA_DIRECTORY / "inmet" / YEAR,
    BRONZE_DIRECTORY / "inmet" / YEAR,
)

station_observations.printSchema()


## 9. Encerrar o Spark

Execute esta célula depois de terminar as ingestões e qualquer inspeção dos DataFrames. Para voltar a usá-los após encerrar a sessão, inicie o Spark novamente e leia os Parquets salvos.


## Borda UTC para concluir 2025

O recorte oficial já foi obtido em `data/inmet/boundary_2025`. Esta célula mantém somente as seis primeiras horas UTC de 01/01/2026. A Gold continua limitada aos dias locais de 2025.

In [ ]:
# Atualiza também a rotina compartilhada de gravação.
reload(bronze_storage)
reload(inmet)

boundary_observations = inmet.ingest_inmet(
    spark,
    DATA_DIRECTORY / "inmet" / f"boundary_{YEAR}",
    BRONZE_DIRECTORY / "inmet" / f"boundary_{YEAR}",
)

boundary_observations.printSchema()


In [ ]:
execution.finish_notebook(session, spark)
